# Chapter 7: What Each Layer Contributes — Dependency

**What you'll learn:** How to compute gradient-based dependency, interpret dependency profiles, and understand the length confound.

**Key concept:** Dependency D_l = ||∂s/∂H̃(l)|| measures how much layer l matters for the model's output. High dependency = that layer's computation affects the prediction.

**Time:** ~20 minutes

## 7.1 Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import layer_time_geometry as core
import ltg

model = ltg.load_model("Qwen/Qwen2.5-7B", device="auto")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

## 7.2 What is Dependency?

After processing a prompt, the model produces a score (log-probability) for the next token. **Dependency** measures how sensitive this score is to the hidden state at each layer — using gradients.

$$D_l = \left\|\frac{\partial s}{\partial \tilde{H}^{(l)}}\right\|_F$$

Layers with high D_l have more influence on the output.

In [2]:
result = ltg.analyse("The Pythagorean theorem states that a squared plus b squared equals c squared", model=model)

# Dependency profile
D = result.dependency_profile
print(f"Dependency profile shape: {D.shape}")
print(f"Total dependency: {result.dep_total:.4f}")
print(f"Dependency entropy: {result.dep_entropy:.3f}")
print(f"Horizon-90: layer {result.dep_horizon_90}")
print(f"Final-3-layer concentration: {result.dep_concentration_final3:.1%}")

# Plot
result.plot_dependency(save_path="ch7_dep.png")
img = plt.imread("ch7_dep.png")
fig, ax = plt.subplots(figsize=(12, 4))
ax.imshow(img); ax.axis('off')
plt.show()

sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.


Dependency profile shape: (29,)
Total dependency: 2.0354
Dependency entropy: 2.503
Horizon-90: layer 16
Final-3-layer concentration: 0.6%
Saved: ch7_dep.png


/var/folders/kj/3h_snmgd70v05_3h0hwqmmkw0000gn/T/ipykernel_61327/2597901403.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7.3 Dependency vs Curvature

Are the layers with high curvature the same as the layers with high dependency? Not always! A layer can be geometrically complex (high curvature) but have low dependency (doesn't affect the output).

In [3]:
# Overlay curvature and dependency profiles
fig, ax1 = plt.subplots(figsize=(10, 5))

curv_norm = result.curvature_by_layer / result.curvature_by_layer.sum()
dep_norm = D / D.sum()

ax1.plot(curv_norm, color='#EE6677', linewidth=2, label='Curvature (normalised)')
ax2 = ax1.twinx()
ax2.plot(dep_norm, color='#4477AA', linewidth=2, label='Dependency (normalised)')

ax1.set_xlabel('Layer', fontsize=12)
ax1.set_ylabel('Normalised curvature', color='#EE6677', fontsize=12)
ax2.set_ylabel('Normalised dependency', color='#4477AA', fontsize=12)
ax1.set_title('Curvature vs Dependency by Layer', fontsize=14)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.show()

# Correlation
corr = np.corrcoef(curv_norm, dep_norm[:len(curv_norm)])[0, 1]
print(f"Correlation between curvature and dependency profiles: {corr:.3f}")

Correlation between curvature and dependency profiles: -0.469


/var/folders/kj/3h_snmgd70v05_3h0hwqmmkw0000gn/T/ipykernel_61327/1429977990.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7.4 The Length Confound

**WARNING:** Total dependency D_total is strongly correlated with sequence length (r ≈ 0.96). This is a mechanical artefact — longer sequences have more hidden states for gradients to flow through.

Always use **normalised** metrics (entropy, horizons, concentration) when comparing prompts of different lengths.

In [4]:
# Demonstrate the length confound
prompts_by_length = [
    "Hi",
    "What is two plus three?",
    "Explain the process of photosynthesis in plants",
    "Describe in detail how the human cardiovascular system works, including the role of the heart, arteries, veins, and capillaries in circulating blood throughout the body",
]

dep_totals = []
entropies = []
lengths = []

for text in prompts_by_length:
    r = ltg.analyse(text, model=model)
    dep_totals.append(r.dep_total)
    entropies.append(r.dep_entropy)
    lengths.append(r.n_tokens)
    print(f"Tokens: {r.n_tokens:3d} | D_total: {r.dep_total:.4f} | Entropy: {r.dep_entropy:.3f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.scatter(lengths, dep_totals, s=100, color='#EE6677')
ax1.set_xlabel('Number of tokens')
ax1.set_ylabel('D_total')
ax1.set_title('D_total vs Length (CONFOUNDED!)')
corr1 = np.corrcoef(lengths, dep_totals)[0, 1]
ax1.annotate(f'r = {corr1:.3f}', xy=(0.05, 0.9), xycoords='axes fraction', fontsize=12, color='red')

ax2.scatter(lengths, entropies, s=100, color='#4477AA')
ax2.set_xlabel('Number of tokens')
ax2.set_ylabel('Dependency entropy')
ax2.set_title('Entropy vs Length (less confounded)')
corr2 = np.corrcoef(lengths, entropies)[0, 1]
ax2.annotate(f'r = {corr2:.3f}', xy=(0.05, 0.9), xycoords='axes fraction', fontsize=12)

plt.tight_layout()
plt.show()
print(f"\nD_total correlation with length: {corr1:.3f} (heavily confounded)")
print(f"Entropy correlation with length: {corr2:.3f} (much less confounded)")

Tokens:   1 | D_total: 0.5368 | Entropy: 1.786


Tokens:   6 | D_total: 7.7911 | Entropy: 2.645


Tokens:   9 | D_total: 4.1061 | Entropy: 2.658


Tokens:  31 | D_total: 0.3168 | Entropy: 2.798

D_total correlation with length: -0.398 (heavily confounded)
Entropy correlation with length: 0.661 (much less confounded)


/var/folders/kj/3h_snmgd70v05_3h0hwqmmkw0000gn/T/ipykernel_61327/686125920.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7.5 Comparing Dependency Across Tasks

In [5]:
task_prompts = {
    'Arithmetic': 'What is 17 times 23?',
    'Logic': 'If all dogs are mammals and Rex is a dog, is Rex a mammal?',
    'Retrieval': 'What is the capital of Japan?',
    'Creative': 'Write a haiku about the moon',
}

fig, ax = plt.subplots(figsize=(10, 5))

for name, text in task_prompts.items():
    r = ltg.analyse(text, model=model)
    D_norm = r.dependency_profile / r.dependency_profile.sum()
    ax.plot(D_norm, linewidth=2, label=f'{name} (H90={r.dep_horizon_90}, S={r.dep_entropy:.2f})')

ax.set_xlabel('Layer', fontsize=12)
ax.set_ylabel('Normalised dependency', fontsize=12)
ax.set_title('Dependency Profiles by Task Type', fontsize=14)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

/var/folders/kj/3h_snmgd70v05_3h0hwqmmkw0000gn/T/ipykernel_61327/586573413.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7.6 Key Takeaways

1. **Dependency** = gradient-based measure of how much each layer matters for the output.
2. **D_total is confounded with length** — always use normalised metrics.
3. **Dependency entropy** measures how spread out the computation is (high = distributed, low = concentrated).
4. **Horizon-90** tells you where the model "makes up its mind".
5. **Dependency tracks task type better than curvature.**

## Exercise

For a fixed prompt, compute the full (L, T) dependency map D_lt using the core library. Which *token position* has the highest total dependency? Is it the last token (where the prediction happens)?

In [6]:
# Your turn!
# text = "What is the capital of France?"
# H_raw = core.extract_hidden_states(model.hf_model, model.tokenizer, text, model.device)
# H_np = H_raw[1:].cpu().numpy()
# metric = core.estimate_metric(H_np, n_components=256)
# dep = core.compute_dependency_density(model.hf_model, model.tokenizer, text, metric, device=model.device)
# D_lt = dep.D_lt  # shape (L, T)
# Plot: plt.imshow(D_lt, aspect='auto', origin='lower')